# 10 · A look-up table upscaler: memory instead of arithmetic

Every model so far computes its answer. This one **looks it up**. A network that only ever sees
four pixels can only be asked so many questions - sampled at every 16th 8-bit value, 17⁴ =
83,521 of them - so after training it answers all of them once, the answers are stored, and the
network is thrown away. Running the model is then: read four pixels, find the cell of the table
they fall in, read five of its corners, and mix them. This is SR-LUT (Jo and Kim, CVPR 2021)
and MuLUT (Li et al., ECCV 2022), on this project's data and in this project's frame.

    LR luma ──► 4 taps, each kernel in 4 rotations ──► look-up: a cell, 5 corners ──► turn back, average ──► HR luma correction
                                                       (a table per kernel: megabytes)
    colour, and the rest of luma: bicubic, as in notebook 7

- **The table is what ships.** 17 points along each tap: every 16th 8-bit value, so a pixel's
  top four bits pick the cell and its bottom four say where in it. One kernel at x2 is 83,521
  rows of 4 sub-pixels - 1.3 MB in float32, 0.67 MB in float16, twenty times the whole of
  notebook 7's graph - and what it costs to run is a few dozen operations per look-up.
- **Five corners, not sixteen.** Simplex interpolation: the order of the four fractions picks
  one of the 24 simplices a 4D cell is cut into, and its five corners are mixed by differences
  of the sorted fractions. It is exact for anything linear in the taps (checked below) and
  continuous from one cell to the next.
- **Rotations widen what one table sees.** A kernel reads four pixels; turned four ways around
  the pixel it answers for, with the same table each time, it covers more - and the model
  turns with its input, exactly (checked). Three kernels (MuLUT's S, D and Y) cover 5x5 for
  three tables, and a second stage on the first one's output 9x9 - the patches' halo of 4.
- **It starts as bicubic**, like every model here: the network's last layer starts at zero,
  and it corrects luma only, as in notebook 7.
- **The address is integer arithmetic.** float16 counts integers only to 2,048, so a row of
  83,521 is found as two numbers under 289, each exact in half precision, that meet in int32.
  Everything else the look-up has to work out - which order, which corners, what weights - is
  small fixed convolutions of small integers.

Trained with the shared recipe (`training.fit`) and scored on notebook 0's validation split beside notebook 7's model.

Needs: `00_data_preprocess.ipynb`, and the scores of notebooks 1 (bicubic), 4 and 8 (notebook 7's model at three
seeds).

In [1]:
"""Settings."""

import math
import os

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

os.environ["DATASETS_DIR"] = "/mnt/d/datasets"   # where the datasets are: set before helpers is imported

from helpers import models, patches, training, webexport
from helpers.training import DATA_DIR, DEVICE

NOTEBOOK = 10
LADDER = {"S": ("S", 1), "SDY": ("SDY", 1), "SDY x2": ("SDY", 2)}   # name -> (kernels, stages)
WIDTH, DEPTH = 64, 4        # the network that fills a table: 4 taps -> 4 layers of 64 -> its answers
LEVELS = 17                 # table points along each tap: every 16th 8-bit value (SR-LUT's)
SWEEP_LEVELS = (9, 17, 33)  # every 32nd, 16th and 8th value: what memory buys
TABLE_LEARNING_RATE = 1e-4  # the tables' own entries, fine-tuned (MuLUT's); the networks train at the shared 1e-3
SEEDS = (0, 1, 2)           # the smallest network trained three times: how far luck alone moves it
RUN_DIR = training.CHECKPOINT_DIR / "lut_runs"

manifest = training.load_manifest()
SCALE = manifest["scale"]
trim = training.trimmer(manifest)   # cuts the halo off an output: the border is padding, not picture
recorded = training.recorded()
bicubic = recorded[1, "bicubic"]    # notebook 1's scores of the bar
luma_24x2 = {name: scores for (notebook, name), scores in recorded.items()
             if notebook in (4, 8) and name.startswith("luma 24 + 2 blocks")}   # notebook 7's model, three seeds


def tag(name):
    return name.replace(" ", "")

## 1. Model

The network and the table share everything but the middle:

- **Taps.** Luma (BT.709, a frozen 1x1 convolution), then, per stage, a frozen convolution that
  copies each kernel's four taps in each of four rotations - a *group* - into four channels. The
  kernels are MuLUT's: **S**, a 2x2 square from the pixel (SR-LUT's only kernel); **D**, the same
  square dilated; **Y**, the pixel and three around its diagonal neighbour. A rotation turns the
  taps around the pixel itself, so S covers 3x3 and D and Y 5x5.
- **Answers.** The *network* answers a group with a small MLP per kernel and stage - 1x1
  convolutions, 4 -> 64 -> 64 -> 64 -> 64 -> its answers - the same in every rotation. The *table*
  holds that MLP's answers at `LEVELS` points along each tap and interpolates between them.
- **Adding up.** In the last stage a group answers the pixel's four x2 sub-pixels as its rotated
  kernel saw them; a frozen 1x1 convolution turns each back, averages the rotations and adds
  the kernels. That is luma's correction, added to R, G and B over bicubic, as in notebook 7.
- **Stages.** An earlier stage answers one number per group, a correction of the LR luma the
  next stage reads. Two stages of SDY reach 4 px, the patches' halo.

The look-up:

- **Where a point is.** Table coordinates are luma in steps of 16/255, so an 8-bit value's top
  four bits are its cell and the bottom four its place in it, and the 17th point (256/255) is
  never reached. `floor` is the cell's lowest corner; the four fractions say where in the cell.
- **Five corners.** Stepping along the taps from the largest fraction to the smallest visits
  the five corners of the simplex the point lies in. Corner k's weight is the k-th largest
  fraction minus the next one (with 1 before the first and 0 after the last). A tie can go
  either way, since both ways give the same value; here it goes to the earlier tap.
- **Bookkeeping as fixed convolutions.** The order is a rank per tap - how many of the other
  three come before it, from six pairwise comparisons - and corner k has stepped along a tap
  once that tap's rank is below k. Differences, ranks, addresses and weights are each a small
  fixed convolution per group, and every value in them but the weights is a small integer,
  exact in float16.
- **An integer address.** A corner's row is two numbers under 289 = 17²: hi from taps 0 and 1,
  lo from taps 2 and 3. They meet in int32 - row = 289 hi + lo, plus where the kernel's table
  starts - and one `Gather` per stage reads every corner of every group.

Untrained, the network's last layer is zero, so the network - and the table it fills - is
bicubic, exactly.

In [2]:
LUMA = torch.tensor(models.LUMA_709)
ROTATIONS = 4
KERNELS = {   # a kernel's four taps, (down, across) from the pixel it answers for
    "S": ((0, 0), (0, 1), (1, 0), (1, 1)),   # SR-LUT's 2x2 square
    "D": ((0, 0), (0, 2), (2, 0), (2, 2)),   # MuLUT: the square, dilated
    "Y": ((0, 0), (1, 1), (1, 2), (2, 1)),   # MuLUT: the diagonal neighbour and two beside it
}
PAIRS = [(i, j) for i in range(4) for j in range(i + 1, 4)]   # the six pairs a sort of four compares


def rotate(offset, turns):
    """An offset (down, across) turned by `turns` quarter turns."""
    dy, dx = offset
    for _ in range(turns):
        dy, dx = dx, -dy
    return dy, dx


def frozen(conv, weight, bias=None):
    """`conv` with these weights (and bias), never trained."""
    with torch.no_grad():
        conv.weight.copy_(weight.reshape(conv.weight.shape))
        if bias is not None:
            conv.bias.copy_(bias)
    for parameter in conv.parameters():
        parameter.requires_grad_(False)
    return conv


def per_group(groups, matrix, bias=None):
    """A frozen 1x1 convolution applying one small matrix (out x in) to every group of channels."""
    out, inputs = matrix.shape
    conv = nn.Conv2d(groups * inputs, groups * out, 1, groups=groups, bias=bias is not None)
    return frozen(conv, matrix.repeat(groups, 1), None if bias is None else bias.repeat(groups))


class FourTapUpscaler(models.Upscaler):
    """Bicubic colour, and luma corrected by stages that each answer from four pixels at a time.

    A stage reads every kernel's four taps in each of four rotations (a group), answers every
    group (`answer`, which a subclass implements) and adds the answers up. Every stage but the
    last corrects the LR luma the next one reads; the last makes the HR luma correction.
    """

    def __init__(self, kernels="S", stages=1, scale=2):
        super().__init__(scale, skip=5)
        self.kernels, self.stages = kernels, stages
        self.groups = len(kernels) * ROTATIONS
        self.outs = [1] * (stages - 1) + [scale * scale]   # what a group answers, per stage
        reach = max(abs(v) for name in kernels for tap in KERNELS[name] for v in tap)
        self.halo = max(stages * reach, self.skip_size // 2)
        self.to_luma = frozen(nn.Conv2d(3, 1, 1, bias=False), LUMA)

        size, centre = 2 * reach + 1, (scale - 1) / 2
        self.reads, self.combines = nn.ModuleList(), nn.ModuleList()
        for out in self.outs:
            # read: channel g * 4 + t is tap t of group g = kernel * ROTATIONS + rotation
            read = torch.zeros(self.groups * 4, 1, size, size)
            for k, name in enumerate(kernels):
                for r in range(ROTATIONS):
                    for t, tap in enumerate(KERNELS[name]):
                        dy, dx = rotate(tap, r)
                        read[(k * ROTATIONS + r) * 4 + t, 0, reach + dy, reach + dx] = 1
            self.reads.append(frozen(nn.Conv2d(1, self.groups * 4, size, padding=reach, bias=False), read))
            # combine: a group's answer o is the sub-pixel o turned by its rotation; rotations are
            # averaged and kernels added
            combine = torch.zeros(out, self.groups * out)
            for g in range(self.groups):
                for o in range(out):
                    dy, dx = rotate((o // scale - centre, o % scale - centre), g % ROTATIONS)
                    to = round(dy + centre) * scale + round(dx + centre) if out > 1 else 0
                    combine[to, g * out + o] = 1 / ROTATIONS
            self.combines.append(frozen(nn.Conv2d(self.groups * out, out, 1, bias=False), combine))

    def answer(self, stage, taps):
        """(N, groups * 4, H, W) taps -> (N, groups * outs[stage], H, W) answers."""
        raise NotImplementedError

    def residual(self, x):
        luma = self.to_luma(x)
        for stage in range(self.stages):
            correction = self.combines[stage](self.answer(stage, self.reads[stage](luma)))
            if stage < self.stages - 1:
                luma = (luma + correction).clamp(0, 1)
        return torch.cat([correction, correction, correction], dim=1)   # on every colour: Y moves


class FourTapNet(FourTapUpscaler):
    """The network that fills the tables: per stage and kernel, an MLP from four taps to a
    group's answers, the same in every rotation. Its last layer starts at zero: bicubic."""

    def __init__(self, kernels="S", stages=1, width=WIDTH, depth=DEPTH, scale=2):
        super().__init__(kernels, stages, scale)
        self.width, self.depth = width, depth
        self.nets = nn.ModuleList()
        for out in self.outs:
            bank = nn.ModuleList()
            for _ in kernels:
                layers, channels = [], 4
                for _ in range(depth):
                    layers += [nn.Conv2d(channels, width, 1), nn.ReLU(inplace=True)]
                    channels = width
                bank.append(nn.Sequential(*layers, models.zero_start(nn.Conv2d(channels, out, 1))))
            self.nets.append(bank)

    def answer(self, stage, taps):
        n, _, h, w = taps.shape
        taps = taps.view(n, len(self.kernels), ROTATIONS * 4, h, w)
        return torch.cat([net(taps[:, k].reshape(n * ROTATIONS, 4, h, w)).view(n, -1, h, w)   # rotations as more pictures
                          for k, net in enumerate(self.nets[stage])], dim=1)


class LutUpscaler(FourTapUpscaler):
    """The same model with every MLP replaced by a table of its answers, `levels` points along
    each tap, read by simplex interpolation. The tables are its only parameters."""

    def __init__(self, kernels="S", stages=1, levels=LEVELS, scale=2):
        super().__init__(kernels, stages, scale)
        L, G = levels, self.groups
        self.levels = levels
        self.spacing = 256 // (levels - 1) / 255   # luma from one table point to the next: 16 / 255 at 17
        for read in self.reads:                     # read table coordinates rather than luma
            read.weight.data /= self.spacing
        self.tables = nn.ParameterList(nn.Parameter(torch.zeros(len(kernels) * L ** 4, out)) for out in self.outs)
        self.register_buffer("first_row", (torch.arange(G, dtype=torch.int32) // ROTATIONS * L ** 4).view(G, 1, 1, 1),
                             persistent=False)   # where each group's kernel's table starts
        self.register_buffer("corners", torch.arange(5.0).view(5, 1, 1, 1), persistent=False)

        # per group: the six differences f_i - f_j, then each tap's rank - how many taps come
        # before it, f_i >= f_j putting i before j and anything else j before i
        differ, rank, rank_bias = torch.zeros(6, 4), torch.zeros(4, 6), torch.zeros(4)
        for p, (i, j) in enumerate(PAIRS):
            differ[p, i], differ[p, j] = 1, -1
            rank[j, p] += 1
            rank[i, p] -= 1
            rank_bias[i] += 1
        # per corner k: its address (hi, lo) from its four coordinates, and S_k, the sum of the k
        # largest fractions; per group: each corner's weight, the k-th largest fraction
        # (S_k - S_k-1) minus the next, with 1 before the first and 0 after the last
        largest = torch.zeros(6, 5)
        for k in range(1, 5):
            largest[k, k], largest[k, k - 1] = 1, -1
        self.differ = per_group(G, differ)
        self.rank = per_group(G, rank, rank_bias)
        self.address = per_group(G * 5, torch.tensor([[L, 1.0, 0, 0], [0, 0, L, 1.0]]))
        self.add_up = per_group(G * 5, torch.ones(1, 4))
        self.weigh = per_group(G, largest[:5] - largest[1:], torch.tensor([1.0, 0, 0, 0, 0]))

    @torch.no_grad()
    def fill_from(self, net):
        """Every table entry becomes the network's answer at that point."""
        level = torch.arange(self.levels, device=self.tables[0].device) * self.spacing
        points = torch.cartesian_prod(level, level, level, level)   # tap 0 slowest, as the address counts
        for stage, bank in enumerate(net.nets):
            for k, mlp in enumerate(bank):
                answers = mlp(points.T[None, :, :, None])[0, :, :, 0].T
                self.tables[stage][k * self.levels ** 4:(k + 1) * self.levels ** 4] = answers
        return self

    def look_up(self, table, coordinates):
        """Simplex interpolation of `table` at (N, groups * 4, H, W) table coordinates in
        [0, levels - 1). Returns (N, groups, H, W, answers)."""
        n, _, h, w = coordinates.shape
        G = self.groups
        base = coordinates.floor()                                       # the cell's lowest corner
        fraction = coordinates - base                                    # where in the cell
        ahead = (self.differ(fraction) >= 0).to(fraction.dtype)          # f_i >= f_j, for each pair i < j
        rank = self.rank(ahead).view(n, G, 1, 4, h, w)                   # how many taps come before tap j
        stepped = (rank < self.corners).to(fraction.dtype)               # (N, G, 5, 4, H, W): corner k has stepped along j
        corner = (base.view(n, G, 1, 4, h, w) + stepped).view(n, G * 20, h, w)
        hi, lo = self.address(corner).view(n, G, 5, 2, h, w).unbind(3)   # each under 289: exact in float16
        row = hi.to(torch.int32) * self.levels ** 2 + lo.to(torch.int32) + self.first_row
        largest = self.add_up((fraction.view(n, G, 1, 4, h, w) * stepped).view(n, G * 20, h, w))
        weight = self.weigh(largest).view(n, G, 5, h, w)                 # adds up to 1
        values = table.index_select(0, row.flatten()).view(n, G, 5, h, w, -1)
        return (values * weight[..., None]).sum(dim=2)

    def answer(self, stage, taps):
        n, _, h, w = taps.shape
        coordinates = taps.clamp(0, self.levels - 1 - 2 ** -6)   # an 8-bit picture never reaches the top
        return self.look_up(self.tables[stage], coordinates).permute(0, 1, 4, 2, 3).reshape(n, -1, h, w)

In [3]:
"""Checks: untrained, every network and table is bicubic; both turn with their input; a table
matches its network on its grid, and gives anything linear back exactly."""

models.check_skip_is_bicubic(SCALE)
probe_lr, _ = training.val_screens(manifest, count=8)
picture = torch.rand(2, 3, 40, 40, device=DEVICE)


@torch.no_grad()
def scrambled(net, seed=0):
    """`net` with every learned weight random, its last layers too, so a check sees more than bicubic."""
    torch.manual_seed(seed)
    for parameter in net.parameters():
        if parameter.requires_grad:
            parameter.normal_(0, 0.3)
    return net


def turned(model, x):
    """How far `model` is from turning with its input: a quarter turn in, against its output turned."""
    return (model(torch.rot90(x, 1, (2, 3))) - torch.rot90(model(x), 1, (2, 3))).abs().max().item()


gaps = {}
with torch.no_grad(), training.full_precision():
    bicubic_out = trim(models.bicubic(probe_lr, SCALE))
    for name, (kernels, stages) in LADDER.items():
        net = FourTapNet(kernels, stages).to(DEVICE)
        assert net.halo <= manifest["halo"], f"{name} reaches further than the patches' halo"
        table = LutUpscaler(kernels, stages).to(DEVICE).fill_from(net)
        gaps[f"{name}: untrained network against bicubic"] = (trim(net(probe_lr)) - bicubic_out).abs().max().item()
        gaps[f"{name}: its table against bicubic"] = (trim(table(probe_lr)) - bicubic_out).abs().max().item()
        table.fill_from(scrambled(net))
        gaps[f"{name}: network, turned"] = turned(net, picture)
        gaps[f"{name}: table, turned"] = turned(table, picture)
        if stages == 1:   # a second stage reads the first one's output, which is on no grid
            greys = (torch.randint(0, LEVELS - 1, (2, 1, 40, 40), device=DEVICE) * table.spacing).expand(-1, 3, -1, -1)
            gaps[f"{name}: table against network, on its grid"] = (table(greys) - net(greys)).abs().max().item()

    # a table of anything linear in the taps gives it back exactly, between the points too - and on ties
    table = LutUpscaler("SDY").to(DEVICE)
    level = torch.arange(LEVELS, dtype=torch.float32, device=DEVICE)
    slopes, offsets = torch.randn(3, 4, 4, device=DEVICE), torch.randn(3, 4, device=DEVICE)   # per kernel
    grid = torch.cartesian_prod(level, level, level, level)
    table.tables[0].copy_(torch.cat([grid @ slopes[k] + offsets[k] for k in range(3)]))
    kernel = torch.arange(table.groups, device=DEVICE) // ROTATIONS
    anywhere = torch.rand(2, table.groups * 4, 10, 10, device=DEVICE) * (LEVELS - 1.01)
    on_ties = (torch.randint(0, LEVELS - 1, anywhere.shape, device=DEVICE)
               + torch.randint(0, 3, anywhere.shape, device=DEVICE) / 4)   # fractions of 0, 1/4 and 1/2
    for label, points in {"anywhere": anywhere, "on ties": on_ties}.items():
        linear = (torch.einsum("ngjhw,gjo->nghwo", points.view(2, table.groups, 4, 10, 10), slopes[kernel])
                  + offsets[kernel][None, :, None, None])
        relative = (table.look_up(table.tables[0], points) - linear).abs().max() / linear.abs().max()
        gaps[f"a linear table, {label} (relative)"] = relative.item()

for name, gap in gaps.items():
    assert gap < 1e-5, f"{name}: {gap:.1e}"
print("exact, to float32 rounding:")
for name, gap in gaps.items():
    print(f"  {name:<44}{gap:.0e}")
print()
for name, (kernels, stages) in LADDER.items():
    net, table = FourTapNet(kernels, stages), LutUpscaler(kernels, stages)
    entries = models.count_parameters(table, True)
    print(f"{name:<8} halo {net.halo}   network {models.count_parameters(net, True):7,} learned parameters   "
          f"table {entries:9,} entries: {entries * 4 / 1e6:4.1f} MB in float32, {entries * 2 / 1e6:4.1f} MB in float16")

exact, to float32 rounding:
  S: untrained network against bicubic        4e-07
  S: its table against bicubic                4e-07
  S: network, turned                          5e-07
  S: table, turned                            7e-07
  S: table against network, on its grid       0e+00
  SDY: untrained network against bicubic      4e-07
  SDY: its table against bicubic              4e-07
  SDY: network, turned                        2e-06
  SDY: table, turned                          2e-06
  SDY: table against network, on its grid     0e+00
  SDY x2: untrained network against bicubic   4e-07
  SDY x2: its table against bicubic           4e-07
  SDY x2: network, turned                     1e-06
  SDY x2: table, turned                       1e-06
  a linear table, anywhere (relative)         2e-07
  a linear table, on ties (relative)          1e-07

S        halo 2   network  13,060 learned parameters   table   334,084 entries:  1.3 MB in float32,  0.7 MB in float16
SDY      halo 2   ne

## 2. Train the networks

`training.fit`, the shared recipe. A network trains as itself, on continuous taps, no table anywhere, and only becomes
a table below. `S` is trained at three seeds, the spread luck alone makes, and `SDY` and `SDY x2` at one.

In [ ]:
def network_key(name, seed=0):
    return f"{name} network" + (f", seed {seed}" if seed else "")


networks, histories = {}, {}
for name, (kernels, stages) in LADDER.items():
    for seed in SEEDS if name == "S" else SEEDS[:1]:
        torch.manual_seed(seed)
        model = FourTapNet(kernels, stages).to(DEVICE)
        histories[network_key(name, seed)], best = training.fit(model, RUN_DIR / f"net_{tag(name)}_seed{seed}.pt",
                                                                manifest, seed=seed, verbose=0)
        networks[network_key(name, seed)] = model.eval()
        print(f"{network_key(name, seed):<20} best epoch {best['epoch']:3} of {len(histories[network_key(name, seed)]):3}")

### Into tables

**Filling.** Every entry of a table becomes its network's answer at that point, and the network
is not used again. Between the points the table is linear, so what filling loses is how far the
network bends between two of them - and in a second stage, also the first stage's error, which
the second one reads.

**What memory buys.** Each network filled at 9, 17 and 33 points along each tap - every 32nd,
16th and 8th 8-bit value. Each step up is twice the points along each of four taps: sixteen
times the table.

In [ ]:
def filled(name, levels=LEVELS):
    """The table of a network on the ladder (seed 0), at `levels` points along each tap."""
    kernels, stages = LADDER[name]
    return LutUpscaler(kernels, stages, levels).to(DEVICE).fill_from(networks[network_key(name)]).eval()


def score(model):
    return training.val_scores(lambda lr, condition: model(lr), manifest)


scores = {key: score(model) for key, model in networks.items()}
sweep = {}
for name in LADDER:
    for levels in SWEEP_LEVELS:
        table = filled(name, levels)
        found = score(table)
        sweep[name, levels] = found["ssim"].mean(), models.count_parameters(table, True) * 2 / 1e6   # float16
        if levels == LEVELS:
            scores[f"{name} table"] = found
        del table

print("validation SSIM-Y of each table; under it, its loss against its network and its size in float16\n")
print(f"{'':<10}{'network':>10}" + "".join(f"{f'{levels} points':>20}" for levels in SWEEP_LEVELS))
for name in LADDER:
    network = scores[network_key(name)]["ssim"].mean()
    print(f"{name:<10}{network:10.4f}" + "".join(f"{sweep[name, levels][0]:20.4f}" for levels in SWEEP_LEVELS))
    print(f"{'':<20}" + "".join(f"{sweep[name, levels][0] - network:+.4f}, {sweep[name, levels][1]:6.1f} MB".rjust(20)
                                for levels in SWEEP_LEVELS))

### Fine-tuning the tables

MuLUT's LUT-aware fine-tuning. The entries are the
parameters: the loss reaches each one through the interpolation weights of the pixels that read
it, and `training.fit` trains them, from the filled table and at 1e-4. A run starts as its
filled table exactly, so whatever it gains is the table learning what its network could not
hand it - the bends between points, and in two stages, what the first stage's table really
answers. Retraining a network means deleting its table's run in `checkpoints/lut_runs/` too.

In [ ]:
tuned = {}
for name in LADDER:
    table = filled(name).train()
    histories[f"{name} table, fine-tuned"], best = training.fit(table, RUN_DIR / f"table_{tag(name)}.pt", manifest,
                                                                learning_rate=TABLE_LEARNING_RATE, verbose=0)
    tuned[name] = table.eval()
    scores[f"{name} table, fine-tuned"] = score(tuned[name])
    print(f"{name} table: best epoch {best['epoch']:3} of {len(histories[f'{name} table, fine-tuned']):3}")

training.plot_history(histories, training.select_bicubic(manifest), key="ssim", loss_name="SSIM-Y + L1 loss")

## 3. Scores

Every network and table on the validation split, beside notebook 7's model at three seeds. **better** / **worse** means
a gap wider than the spread of the `S` network's three seeds; **luck** means this set cannot tell the two apart.

The questions, in order: what each table loses against the network it was filled from, what fine-tuning wins back,
what each step up the ladder buys (one table seeing 3×3, three seeing 5×5, two stages seeing 9×9), and how far notebook
7's model still is.

In [ ]:
for name, s in scores.items():
    training.record(NOTEBOOK, f"LUT {name}", s)
training.show(training.compare({"bicubic": bicubic, **luma_24x2, **{f"LUT {n}": s for n, s in scores.items()}},
                               manifest, base=bicubic))

mean = lambda name: scores[name]["ssim"].mean()
spread = np.ptp([mean(network_key("S", seed)) for seed in SEEDS])
print(f"\nS network at {len(SEEDS)} seeds: {spread:.4f} SSIM-Y apart")


def verdict(a, b, label):
    word = "luck" if abs(a - b) <= spread else ("better" if a > b else "worse")
    print(f"{label:<50}{a - b:+.4f} SSIM-Y  {word}")


best_name = max(LADDER, key=lambda name: mean(f"{name} table, fine-tuned"))
for name in LADDER:
    verdict(mean(f"{name} table"), mean(network_key(name)), f"{name}: its table against its network")
    verdict(mean(f"{name} table, fine-tuned"), mean(f"{name} table"), f"{name}: fine-tuned over filled")
    verdict(mean(f"{name} table, fine-tuned"), bicubic["ssim"].mean(), f"{name}: fine-tuned table over bicubic")
names = list(LADDER)
for smaller, larger in zip(names, names[1:]):
    verdict(mean(f"{larger} table, fine-tuned"), mean(f"{smaller} table, fine-tuned"),
            f"{larger} over {smaller}, fine-tuned tables")
verdict(float(np.mean([s["ssim"].mean() for s in luma_24x2.values()])), mean(f"{best_name} table, fine-tuned"),
        f"notebook 7's model (mean of its seeds) over the {best_name} table")

### How much of a table pictures read

A table spends the same memory on every combination of four values, and pictures are not
random: neighbouring pixels are mostly alike. For each fine-tuned table and kernel, the share of
its cells any validation pixel reads, and the share that holds 99% of the reads. Then, for the
`S` table, reads and cells by the *spread* of a cell - its largest coordinate minus its smallest,
in table steps - added up from 0: where the reads are against where the memory is. If nearly
every read lands at small spreads and most cells do not, most of the memory describes
neighbourhoods that do not occur, and a table kept fine near its diagonal and coarse elsewhere
would be far smaller - the idea of diagonal-first compression of these tables (Li et al., 2024).

In [ ]:
@torch.no_grad()
def reads(model):
    """How many times validation pixels (the cores) land in each cell of each stage's table."""
    counts = [torch.zeros(len(model.kernels) * model.levels ** 4, dtype=torch.long, device=DEVICE) for _ in model.tables]
    place = torch.tensor([model.levels ** p for p in (3, 2, 1, 0)], device=DEVICE).view(1, 1, 4, 1, 1)
    halo = manifest["halo"]
    for lr, _ in val_loader:
        luma = model.to_luma(lr.to(DEVICE))
        for stage in range(model.stages):
            taps = model.reads[stage](luma)
            cell = taps.clamp(0, model.levels - 1 - 2 ** -6).floor()[..., halo:-halo, halo:-halo]
            n, _, h, w = cell.shape
            row = (cell.view(n, model.groups, 4, h, w) * place).sum(2).long() + model.first_row.long().view(1, -1, 1, 1)
            counts[stage] += torch.bincount(row.flatten(), minlength=len(counts[stage]))
            luma = (luma + model.combines[stage](model.answer(stage, taps))).clamp(0, 1)
    return counts


val_loader = DataLoader(patches.PatchPairs(DATA_DIR, "val", manifest, augment=False), batch_size=256)
shares = {}   # (ladder rung, stage, kernel) -> the share of all reads each cell of that table gets
for name in LADDER:
    model = tuned[name]
    for stage, count in enumerate(reads(model)):
        for k, kernel in enumerate(model.kernels):
            share = count[k * model.levels ** 4:(k + 1) * model.levels ** 4].double()
            share /= share.sum()
            shares[name, stage, kernel] = share
            used = (share > 0).double().mean().item()
            hold = (share.sort(descending=True).values.cumsum(0) < 0.99).double().mean().item()
            print(f"{name:<8} stage {stage + 1}  kernel {kernel}:  {used:6.1%} of the cells read at all, "
                  f"99% of the reads in {hold:6.2%}")

# the S table: reads and cells by spread, added up from 0
level = torch.arange(LEVELS, device=DEVICE)
grid = torch.cartesian_prod(level, level, level, level)
spread_of = grid.max(1).values - grid.min(1).values   # a cell's lowest corner stops at LEVELS - 2
inside = grid.max(1).values < LEVELS - 1              # rows that are a cell, not the table's far edge
spreads = range(LEVELS - 1)
s_share = shares["S", 0, "S"]
read_share = np.cumsum([s_share[(spread_of == s) & inside].sum().item() for s in spreads])
cell_count = np.cumsum([((spread_of == s) & inside).sum().item() for s in spreads])
cell_share = cell_count / cell_count[-1]

fig, ax = plt.subplots(figsize=(7.5, 3.8))
ax.plot(spreads, read_share * 100, marker="o", label="reads")
ax.plot(spreads, cell_share * 100, marker="o", label="cells (memory)")
ax.set(xlabel="spread of a cell: its largest coordinate minus its smallest, in table steps",
       ylabel="share up to that spread, %", title="S table: where the reads are, against where the memory is",
       xticks=list(spreads), ylim=(0, 102))
ax.grid(color="0.9")
ax.legend(loc="lower right")
fig.tight_layout()
plt.show()
for s in (0, 1, 2, 4):
    print(f"spread <= {s}: {read_share[s]:7.2%} of the reads, {cell_share[s]:7.2%} of the cells")

### What it looks like

Four validation screens at a slice QP under 26: bicubic, the best fine-tuned table and the target.

In [ ]:
lr, hr = training.val_screens(manifest)
with torch.no_grad():
    training.show_patches({"bicubic": trim(models.bicubic(lr, SCALE)), f"{best_name} table": trim(tuned[best_name](lr)),
                           "target": hr})

## 4. Export for the browser

The best fine-tuned table, and the `S` table if it is not the best (one table, the cheapest tier there is), at the
client's tile and as a whole 1080p frame, in float32 and float16, each checked against torch and timed on this CPU. A
table's graph reshapes its look-up at the size it was exported at, so the whole-frame graph is exported at the frame's
size rather than reshaped from the tile's. Besides `Conv`, `Add`, `Concat` and `DepthToSpace`, a table's graph needs
`Floor`, `Clip`, comparisons, `Cast`, integer arithmetic and the `Gather` that reads the table: all outside the
browser's fast set.

In [ ]:
for name in dict.fromkeys([best_name, "S"]):   # one entry if the best is S
    module = tuned[name]
    kernels, stages = LADDER[name]
    slug = f"lut_{kernels.lower()}" + (f"_x{stages}" if stages > 1 else "") + f"_{module.levels}"
    webexport.publish(module, f"upscale_{slug}", module.halo, label=slug, scale=SCALE, float16=True, reshape=False)
print("compare in a browser: uv run src/upscale/benchmark/main.py")

## 5. What it says

*(written after the run)*